In [5]:
!pip install shap  

In [9]:
import shap
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [10]:
import shap
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("diabetes_cleaned.csv")

X = df.drop("Diabetes", axis=1)
y = df["Diabetes"]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Load model
model = joblib.load("random_forest.pkl")

# Create explainer
explainer = shap.TreeExplainer(model)

# Calculate SHAP values
shap_values = explainer.shap_values(X_test)

print(type(shap_values))
print(type(shap_values[0]))

try:
    print(shap_values.shape)
except:
    print("No shape")

print("Length:", len(shap_values))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
(51, 16, 2)
Length: 51


In [ ]:

# SHAP ANALYSIS FOR RANDOM FOREST




# Load Dataset

df = pd.read_csv("diabetes_cleaned.csv")

X = df.drop("Diabetes", axis=1)
y = df["Diabetes"]


# Train-Test Split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# Load Saved Random Forest Model
model = joblib.load("random_forest.pkl")

print("Model Loaded Successfully!")


# Create SHAP Explainer
explainer = shap.TreeExplainer(model)


# Calculate SHAP Values
shap_values = explainer.shap_values(X_test)

print("SHAP Values Shape:")
print(np.array(shap_values).shape)


# Select Diabetes Class (Class = 1)
diabetes_shap = shap_values[:, :, 1]


# Create SHAP Explanation Object
explanation = shap.Explanation(
    values=diabetes_shap,
    base_values=np.repeat(explainer.expected_value[1], len(X_test)),
    data=X_test.values,
    feature_names=X_test.columns.tolist()
)


# GLOBAL FEATURE IMPORTANCE
print("\nGenerating SHAP Bar Plot...")

shap.plots.bar(
    explanation,
    max_display=16
)


# SHAP BEESWARM


print("\nGenerating Beeswarm Plot...")

shap.plots.beeswarm(
    explanation,
    max_display=16
)

# ============================================================
# WATERFALL PLOT
# ============================================================

patient = 0

print("\nGenerating Waterfall Plot...")

shap.plots.waterfall(
    explanation[patient],
    max_display=16
)

# ============================================================
# FORCE PLOT
# ============================================================

shap.initjs()

shap.plots.force(
    explanation[patient]
)

# ============================================================
# DEPENDENCE PLOT
# ============================================================

print("\nGenerating Dependence Plot...")

shap.plots.scatter(
    explanation[:, "Age"]
)

# ============================================================
# FEATURE CONTRIBUTION TABLE
# ============================================================

contribution = pd.DataFrame({

    "Feature": X.columns,

    "Value": X_test.iloc[patient].values,

    "SHAP Value": explanation.values[patient]

})

contribution["Absolute SHAP"] = contribution["SHAP Value"].abs()

contribution = contribution.sort_values(
    by="Absolute SHAP",
    ascending=False
)

print("\nFeature Contributions")

print(contribution)

# ============================================================
# CONVERT TO PERCENTAGE
# ============================================================

contribution["Contribution (%)"] = (
    contribution["Absolute SHAP"]
    / contribution["Absolute SHAP"].sum()
) * 100

print("\nContribution Percentage")

print(
    contribution[
        ["Feature","Contribution (%)"]
    ]
)

# ============================================================
# SAVE CSV
# ============================================================

contribution.to_csv(
    "patient_feature_contribution.csv",
    index=False
)

print("\nCSV Saved Successfully!")

# ============================================================
# PREDICTION
# ============================================================

prediction = model.predict(
    X_test.iloc[[patient]]
)[0]

probability = model.predict_proba(
    X_test.iloc[[patient]]
)[0][1]

print("\nPrediction")

if prediction == 1:
    print("Diabetes Detected")
else:
    print("No Diabetes")

print(f"\nProbability : {probability*100:.2f}%")

print("\nSHAP Analysis Completed Successfully!")